In [2]:
import os
import shutil
import numpy as np

import keras

from delft.sequenceLabelling import Sequence
from delft.sequenceLabelling.config import ModelConfig
from delft.sequenceLabelling.reader import load_data_and_labels_crf_file
from delft.sequenceLabelling.models import get_model
from delft.sequenceLabelling.preprocess import Preprocessor
from delft.utilities.Embeddings import Embeddings
from delft.sequenceLabelling.wrapper import CONFIG_FILE_NAME, PROCESSOR_FILE_NAME

2025-09-04 14:48:02.060240: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-04 14:48:02.073320: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-09-04 14:48:02.190724: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-09-04 14:48:02.303069: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756954082.390634 1166165 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756954082.41

In [3]:
model_name = "grobid-date-BidLSTM_CRF"
dir_path = "data/models/sequenceLabelling/"
model_path = os.path.join(dir_path, model_name)
out_dir = "./export/"
input_data_path = "./data/sequenceLabelling/grobid/date/date-110322.train"
limit = 32
# Load data
x_all, y_all, f_all = load_data_and_labels_crf_file(input_data_path)
x_all = x_all[:limit]
y_all = y_all[:limit]
f_all = f_all[:limit] if f_all is not None else None


In [4]:
# Load model and get inner/base models
seq = Sequence(model_name)

# get the model config and overwrite the model config in seq
dir_path = 'data/models/sequenceLabelling/'
model_path = os.path.join(dir_path, model_name)
model_config = ModelConfig.load(os.path.join(model_path, CONFIG_FILE_NAME))
seq.model_config = model_config

seq.embeddings = Embeddings(
    model_config.embeddings_name, resource_registry=seq.registry, use_ELMo=model_config.use_ELMo)

seq.p = Preprocessor.load(os.path.join(dir_path, model_config.model_name, PROCESSOR_FILE_NAME))

seq.model = get_model(
    model_config, seq.p, ntags=len(seq.p.vocab_tag), load_pretrained_weights=False, local_path=model_path)

BidLSTM_CRF


2025-09-04 14:48:19.071492: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [5]:
seq.model.summary(line_length=100, expand_nested=True, show_trainable=True)
seq.model.base_model.summary(line_length=100, expand_nested=True, show_trainable=True)
# seq.model.crf.summary(line_length=120, expand_nested=True, show_trainable=True)

Model: "crf_wrapper"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Layer (type)                          ┃ Output Shape                  ┃        Param # ┃ Traina… ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ functional (Functional)               │ (None, None, 100)             │        392,850 │    Y    │
├───────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────┤
│    └ char_input (InputLayer)          │ (None, None, 30)              │              0 │    -    │
├───────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────┤
│    └ time_distributed                 │ (None, None, 30, 25)          │          1,750 │    Y    │
│ (TimeDistributed)                     │                               │                │         │
├───────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────┤
│    └ word_input (InputLayer)          │ (None, None, 300)             │              0 │    -    │
├───────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────┤
│    └ time_distributed_1               │ (None, None, 50)              │         10,200 │    Y    │
│ (TimeDistributed)                     │                               │                │         │
├───────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────┤
│    └ concatenate (Concatenate)        │ (None, None, 350)             │              0 │    -    │
├───────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────┤
│    └ dropout (Dropout)                │ (None, None, 350)             │              0 │    -    │
├───────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────┤
│    └ bidirectional_1 (Bidirectional)  │ (None, None, 200)             │        360,800 │    Y    │
├───────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────┤
│    └ dropout_1 (Dropout)              │ (None, None, 200)             │              0 │    -    │
├───────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────┤
│    └ dense (Dense)                    │ (None, None, 100)             │         20,100 │    Y    │
├───────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────┤
│    └ length_input (InputLayer)        │ (None, 1)                     │              0 │    -    │
├───────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────┤
│    └ length_passthrough (TakeFirst)   │ (None, None, 100)             │              0 │    -    │
├───────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────┤
│ crf (CRF)                             │ [(None, None), (None, None,   │             63 │    Y    │
│                                       │ 7), (None), (7, 7)]           │                │         │
└───────────────────────────────────────┴───────────────────────────────┴────────────────┴─────────┘

 Total params: 392,913 (1.50 MB)

 Trainable params: 392,913 (1.50 MB)

 Non-trainable params: 0 (0.00 B)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Layer (type)             ┃ Output Shape         ┃       Param # ┃ Connected to         ┃ Traina… ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ char_input (InputLayer)  │ (None, None, 30)     │             0 │ -                    │    -    │
├──────────────────────────┼──────────────────────┼───────────────┼──────────────────────┼─────────┤
│ time_distributed         │ (None, None, 30, 25) │         1,750 │ char_input[0][0]     │    Y    │
│ (TimeDistributed)        │                      │               │                      │         │
├──────────────────────────┼──────────────────────┼───────────────┼──────────────────────┼─────────┤
│ word_input (InputLayer)  │ (None, None, 300)    │             0 │ -                    │    -    │
├──────────────────────────┼──────────────────────┼───────────────┼──────────────────────┼─────────┤
│ time_distributed_1       │ (None, None, 50)     │        10,200 │ time_distributed[0]… │    Y    │
│ (TimeDistributed)        │                      │               │                      │         │
├──────────────────────────┼──────────────────────┼───────────────┼──────────────────────┼─────────┤
│ concatenate              │ (None, None, 350)    │             0 │ word_input[0][0],    │    -    │
│ (Concatenate)            │                      │               │ time_distributed_1[… │         │
├──────────────────────────┼──────────────────────┼───────────────┼──────────────────────┼─────────┤
│ dropout (Dropout)        │ (None, None, 350)    │             0 │ concatenate[0][0]    │    -    │
├──────────────────────────┼──────────────────────┼───────────────┼──────────────────────┼─────────┤
│ bidirectional_1          │ (None, None, 200)    │       360,800 │ dropout[0][0]        │    Y    │
│ (Bidirectional)          │                      │               │                      │         │
├──────────────────────────┼──────────────────────┼───────────────┼──────────────────────┼─────────┤
│ dropout_1 (Dropout)      │ (None, None, 200)    │             0 │ bidirectional_1[0][… │    -    │
├──────────────────────────┼──────────────────────┼───────────────┼──────────────────────┼─────────┤
│ dense (Dense)            │ (None, None, 100)    │        20,100 │ dropout_1[0][0]      │    Y    │
├──────────────────────────┼──────────────────────┼───────────────┼──────────────────────┼─────────┤
│ length_input             │ (None, 1)            │             0 │ -                    │    -    │
│ (InputLayer)             │                      │               │                      │         │
├──────────────────────────┼──────────────────────┼───────────────┼──────────────────────┼─────────┤
│ length_passthrough       │ (None, None, 100)    │             0 │ dense[0][0],         │    -    │
│ (TakeFirst)              │                      │               │ length_input[0][0]   │         │
└──────────────────────────┴──────────────────────┴───────────────┴──────────────────────┴─────────┘

 Total params: 392,850 (1.50 MB)

 Trainable params: 392,850 (1.50 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
# ok - now we have the model created but probably not in the graph-mode sense
# let's try and save it anyway
os.makedirs(os.path.join(out_dir, model_config.model_name), exist_ok=True)
keras.saving.save_model(
    seq.model.base_model, os.path.join(out_dir, model_config.model_name, 'unmade_base_model.keras'))
keras.saving.save_model(
    seq.model.crf, os.path.join(out_dir, model_config.model_name, 'unmade_crf.keras'))
# this calls save weights on the base_model.model and, I assume, the CRF layer
seq.model.save(os.path.join(out_dir, model_config.model_name, 'unmade_model.weights.h5'))

In [8]:
editor_base = keras.saving.KerasFileEditor(os.path.join(out_dir, model_config.model_name, 'unmade_base_model.keras'))
editor_base.summary()

editor_crf = keras.saving.KerasFileEditor(os.path.join(out_dir, model_config.model_name, 'unmade_crf.keras'))
editor_crf.summary()

editor_model = keras.saving.KerasFileEditor(os.path.join(out_dir, model_config.model_name, 'unmade_model.weights.h5'))
editor_model.summary()


Keras model file './export/grobid-date-BidLSTM_CRF/unmade_base_model.keras'

Saved with Keras 3.10.0 - date: 2025-09-04@14:49:43

Weights structure

> layers

└─ dense

...

Weights structure

> crf

├─ 0: shape=(7, 7), dtype=float32

├─ 1: shape=(7,), dtype=float32

└─ 2: shape=(7,), dtype=float32

In [38]:
# copy the model weights file to the export directory
shutil.copy("./data/models/sequenceLabelling/grobid-date-BidLSTM_CRF/model_weights.hdf5", "./export/grobid-date-BidLSTM_CRF/legacy_model.weights.h5")
legacy_model_editor = keras.saving.KerasFileEditor("./export/grobid-date-BidLSTM_CRF/legacy_model.weights.h5")

Keras model file './export/grobid-date-BidLSTM_CRF/legacy_model.weights.h5'

In [39]:
legacy_model_editor.summary()

Weights structure

> crf

│   ├─ chain_kernel:0: shape=(7, 7), dtype=float32

│   ├─ crf

│   │   └─ dense_1

...

└─ forward_lstm

└─ lstm_cell_1

├─ bias:0: shape=(100,), dtype=float32

├─ kernel:0: shape=(25, 100), dtype=float32

└─ recurrent_kernel:0: shape=(25, 100), dtype=float32

In [42]:
legacy_model_editor.compare(seq.model.model)

Running comparison

...Object /base_model present in reference model, missing from saved file

In reference model, /base_model contains the following keys: ['layers']

...Weight /crf/0 present in reference model, missing from saved file

...Weight /crf/1 present in reference model, missing from saved file

...

...Object /model present in saved file, missing from reference model

In saved file, /model contains the following keys: ['bidirectional_1', 'dense', 'time_distributed', 
'time_distributed_1']

─────────────────────

Found 11 errors: saved file is not compatible with the reference model

{'status': 'error', 'error_count': 11, 'match_count': 0}

In [10]:
[(v.path, v.shape, v.dtype) for v in seq.model.weights]

[('time_distributed/char_embeddings/embeddings',
  TensorShape([70, 25]),
  'float32'),
 ('time_distributed_1/bidirectional/forward_lstm/lstm_cell/kernel',
  TensorShape([25, 100]),
  'float32'),
 ('time_distributed_1/bidirectional/forward_lstm/lstm_cell/recurrent_kernel',
  TensorShape([25, 100]),
  'float32'),
 ('time_distributed_1/bidirectional/forward_lstm/lstm_cell/bias',
  TensorShape([100]),
  'float32'),
 ('time_distributed_1/bidirectional/backward_lstm/lstm_cell/kernel',
  TensorShape([25, 100]),
  'float32'),
 ('time_distributed_1/bidirectional/backward_lstm/lstm_cell/recurrent_kernel',
  TensorShape([25, 100]),
  'float32'),
 ('time_distributed_1/bidirectional/backward_lstm/lstm_cell/bias',
  TensorShape([100]),
  'float32'),
 ('bidirectional_1/forward_lstm_1/lstm_cell/kernel',
  TensorShape([350, 400]),
  'float32'),
 ('bidirectional_1/forward_lstm_1/lstm_cell/recurrent_kernel',
  TensorShape([100, 400]),
  'float32'),
 ('bidirectional_1/forward_lstm_1/lstm_cell/bias',
  Te

In [11]:
seq.model.save_weights(os.path.join(out_dir, model_config.model_name, 'upgraded_model.weights.h5'))

In [ ]:

inner_model = getattr(seq.model, 'model', seq.model)
base_model = getattr(inner_model, 'base_model', None)
if base_model is None:
    raise RuntimeError('No base_model found')

In [18]:
# Build generator
gen = seq.model.get_generator()
test_gen = gen(
    x_all, y_all,
    batch_size=min(limit, seq.model_config.batch_size),
    preprocessor=seq.p,
    char_embed_size=seq.model_config.char_embedding_size,
    max_sequence_length=seq.model_config.max_sequence_length,
    embeddings=seq.embeddings,
    shuffle=False,
    features=f_all,
    output_input_offsets=True,
    use_chain_crf=seq.model_config.use_chain_crf,
)

data, label = test_gen[0]
model_inputs = data

model_output = seq.model.model(model_inputs)

# try saving the model again
keras.saving.save_model(
    seq.model.model, os.path.join(out_dir, model_config.model_name, 'unmade_model.keras'))

editor_post_eval = keras.saving.KerasFileEditor(os.path.join(out_dir, model_config.model_name, 'unmade_model.keras'))
editor_post_eval.summary()


Keras model file './export/grobid-date-BidLSTM_CRF/unmade_model.keras'

Saved with Keras 3.10.0 - date: 2025-09-03@13:36:48

Weights structure

> crf

├─ 0: shape=(7, 7), dtype=float32

├─ 1: shape=(7,), dtype=float32

└─ 2: shape=(7,), dtype=float32

In [ ]:

# Run base model to get pre-CRF features
feats = base_model.predict_on_batch(model_inputs)

out = {
    'feats': tensor_summary(feats),
}

# Optional: extract a few positions where tokens look numeric (potential month/day confusions)
numeric_positions = []
num_pat = re.compile(r'^[0-9]{1,4}$')
# length_input is last
import numpy as np
seq_lengths = np.reshape(model_inputs[-1], (-1,))
B, T, F = feats.shape
for b in range(min(B, len(x_all))):
    L = int(seq_lengths[b]) if b < len(seq_lengths) else T
    tokens = x_all[b]
    for t in range(min(L, len(tokens))):
        tok = tokens[t]
        if num_pat.match(tok):
            # record feature vector norm for a small sample
            v = feats[b, t]
            numeric_positions.append({
                'b': b,
                't': t,
                'token': tok,
                'feat_l2': float(np.linalg.norm(v)),
                'feat_min': float(np.min(v)),
                'feat_max': float(np.max(v)),
                'feat_mean': float(np.mean(v)),
                'feat_std': float(np.std(v)),
            })
            if len(numeric_positions) >= 40:
                break
    if len(numeric_positions) >= 40:
        break

out['numeric_positions'] = numeric_positions

with open(os.path.join(out_dir, model_config.model_name, 'numeric_positions.json'), 'w', encoding='utf-8') as f:
    json.dump(out, f, indent=2)
